# VocEd Lab 9 — U-Net

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab9_unet.ipynb)

Lab 8's MinimalCNN learned its own kernels, but each prediction could only see a 5 × 5 window —
far smaller than a nucleus.  A **U-Net** sees much further:

- an **encoder** shrinks the image step by step (256 → 128 → 64), so the same 3 × 3 kernels cover
  a wider and wider area;
- a **decoder** grows it back to 256 × 256, so every pixel still gets a label;
- **skip connections** hand the fine detail from each encoder level straight to the decoder.

The book reports a surprise: on this data, its U-Net scored *below* the MinimalCNN, and it names
the likely causes — too many weights for 120 training images, and no **data augmentation**.  We
test both claims.

**Textbook:** [Chapter 8, Part 2 — U-Net for Cell Segmentation](https://obooks.tech/imageprocessing/chapter8.html#part-2-u-net-for-cell-segmentation).

---

### ⚡ Use a GPU

In Colab: **Runtime → Change runtime type → T4 GPU**, then run from the top.  On a GPU each training
run takes a few minutes.  On a CPU it takes about an hour — use a GPU for this lab.

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

## 0. Setup — Get the Data

The same images and split as Labs 3–8.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)
assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–8"

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

## 1. The Architecture

The building block is a `ConvBlock`: two 3 × 3 convolutions, each followed by **batch
normalisation** (which keeps the values in a steady range and helps training) and ReLU.

| Level | Encoder | Size | Decoder |
|---|---|---|---|
| 1 | `enc1`: 3 → 32 maps | 256 × 256 | `dec1`: (64 + 32) → 32, then a 1 × 1 conv → 3 classes |
| 2 | `enc2`: 32 → 64 maps | 128 × 128 | `dec2`: (128 + 64) → 64 |
| bottom | `bottleneck`: 64 → 128 maps | 64 × 64 | |

Going down, `MaxPool2d(2)` keeps the largest of every 2 × 2 block, halving the size.  Going up,
`F.interpolate` doubles it.  At each decoder level, `torch.cat` glues the encoder's maps from the same
level next to the upsampled ones — that is the **skip connection**.

📖 Book: [§8.3.7 The U-Net Architecture](https://obooks.tech/imageprocessing/chapter8.html#the-u-net-architecture)

### 🧩 Puzzle 1 — Channels at the junction

`dec2` receives the bottleneck's 128 maps, upsampled, *plus* the skip from `enc2`.  How many
input maps is that?

<details><summary><b>💡 Hint</b> (click to open)</summary>

Count the maps `enc2` produces.

</details>

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 32)
        self.enc2 = ConvBlock(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(64, 128)
        self.dec2 = ConvBlock(128 + ___, 64)
        self.dec1 = ConvBlock(64 + 32, 32)
        self.out_conv = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)                                      # (N,  32, 256, 256)
        s2 = self.enc2(self.pool(s1))                          # (N,  64, 128, 128)
        b  = self.bottleneck(self.pool(s2))                    # (N, 128,  64,  64)
        u2 = F.interpolate(b, scale_factor=2, mode='bilinear', align_corners=False)
        u2 = self.dec2(torch.cat([u2, s2], dim=1))             # (N,  64, 128, 128)
        u1 = F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False)
        u1 = self.dec1(torch.cat([u1, s1], dim=1))             # (N,  32, 256, 256)
        return self.out_conv(u1)                               # (N,   3, 256, 256)

out = SmallUNet()(torch.zeros(1, 3, 256, 256))
n_weights = sum(p.numel() for p in SmallUNet().parameters())
print("Output shape:", tuple(out.shape))
print(f"Numbers to learn: {n_weights:,}  (MinimalCNN: 5,187 — about {n_weights / 5187:.0f} times fewer)")

# ── Check ──
assert out.shape == (1, 3, 256, 256), "128 upsampled maps plus the 64 maps of the enc2 skip"
print("✅ Puzzle 1 solved!")

### 🤔 Think 1 — How far can it see?

1. In the bottleneck, the image is 64 × 64: every pixel there stands for a 4 × 4 block of the
   original.  How wide an area of the original does a 3 × 3 window at the bottleneck cover?
2. Is that enough to see a whole nucleus (30–60 pixels across)?  *(The convolutions above and below
   the bottleneck add more; the book puts the total at about 44 × 44.)*
3. Downsampling throws away detail — a 64 × 64 map cannot draw a sharp 256 × 256 border.  Which part
   of the U-Net brings that detail back?

📖 Book: [§8.3.2 What “Receptive Field” Means](https://obooks.tech/imageprocessing/chapter8.html#what-receptive-field-means) and
[§8.3.8 Skip Connections, Intuitively](https://obooks.tech/imageprocessing/chapter8.html#skip-connections-intuitively)

✏️ **Your answer:** *(double-click this cell and type here)*

## 2. The Loss and the Training Loop

The loss adds two parts:

- **cross-entropy**, as in Lab 8 — scored pixel by pixel;
- **soft Dice loss** — `1 − Dice`, computed from the predicted *probabilities* so that it has
  gradients.  It scores the overlap of whole regions, which is what we report.

📖 Book: [§8.3.9 Loss and Metrics for Segmentation](https://obooks.tech/imageprocessing/chapter8.html#loss-and-metrics-for-segmentation)

In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    probs  = torch.softmax(logits, dim=1)                                  # (N, 3, H, W)
    onehot = F.one_hot(targets, 3).permute(0, 3, 1, 2).float()              # (N, 3, H, W)
    loss = 0.0
    for c in range(3):
        intersection = (probs[:, c] * onehot[:, c]).sum()
        loss += 1 - (2 * intersection + smooth) / (probs[:, c].sum() + onehot[:, c].sum() + smooth)
    return loss / 3

ce = nn.CrossEntropyLoss()
def combined_loss(logits, targets):
    return ce(logits, targets) + dice_loss(logits, targets)

### Data augmentation

We have only 120 training images.  But a cell flipped left to right, or turned by 90°, is still a
perfectly good cell, with a perfectly good mask.  **Augmentation** shows the network a randomly
flipped and rotated copy of each image every time it is used — a different copy each epoch.

### 🧩 Puzzle 2 — Flip the mask with the image

The image has shape `(3, 256, 256)`: its columns are dimension 2.  The mask has shape `(256, 256)`:
which dimension are its columns?  Image and mask must get exactly the same flip, or the labels no
longer match the pixels.

📖 Book: [§8.3.11 Why 0.77 — and How to Push It Higher](https://obooks.tech/imageprocessing/chapter8.html#why-0.77-and-how-to-push-it-higher)

<details><summary><b>💡 Hint</b> (click to open)</summary>

The mask has no colour dimension in front, so every dimension number is one less than the image's.

</details>

In [ ]:
class SegDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = indices
        self.augment = augment
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        image = torch.from_numpy(X[i])                         # (3, 256, 256)
        mask  = torch.from_numpy(y[i].astype(np.int64))        # (256, 256)
        if self.augment:
            if torch.rand(1) < 0.5:                            # flip left ↔ right
                image = torch.flip(image, dims=[2])
                mask  = torch.flip(mask,  dims=[___])
            turns = int(torch.randint(0, 4, (1,)))             # turn by 0, 90, 180 or 270 degrees
            image = torch.rot90(image, turns, dims=[1, 2])
            mask  = torch.rot90(mask,  turns, dims=[0, 1])
        return image, mask

torch.manual_seed(3)
img, msk = SegDataset([7], augment=True)[0]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(img.numpy().transpose(1, 2, 0))
axes[0].set_title('augmented image 7')
axes[1].imshow(msk.numpy(), cmap='viridis', vmin=0, vmax=2)
axes[1].set_title('its mask')
for ax in axes:
    ax.axis('off')
plt.show()

# ── Check ──
torch.manual_seed(3)
a, b = SegDataset([7], augment=True)[0]
dark = (a.numpy().mean(axis=0) < 0.35)
assert (b.numpy()[dark] == 2).mean() > 0.3, "The mask must be flipped the same way as the image"
print("✅ Puzzle 2 solved!")

The training loop is Lab 8's, plus one thing: after every epoch we score the **validation**
images, so we can watch for overfitting and keep the epoch that did best.

In [ ]:
def predict_masks(model, indices):
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(indices), 8):
            batch = torch.from_numpy(X[indices[start:start + 8]]).to(device)
            preds.append(model(batch).argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)

def mean_dice(preds, indices):
    return np.mean([image_dice(p, y[i]) for p, i in zip(preds, indices)])

def train(model, augment, epochs=20):
    torch.manual_seed(0)
    loader = DataLoader(SegDataset(train_idx, augment), batch_size=4, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    history, best, best_state = [], -1, None
    for epoch in range(epochs):
        model.train()
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            loss = combined_loss(model(images), masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        val = mean_dice(predict_masks(model, val_idx), val_idx)
        history.append(val)
        if val > best:
            best, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  epoch {epoch + 1:2d}: validation Dice {val:.4f}")
    model.load_state_dict(best_state)          # keep the best epoch
    return history

## 3. Three Training Runs

1. the U-Net as in the book — no augmentation;
2. the same U-Net **with augmentation**;
3. a U-Net **without skip connections** (with augmentation), to see what the skips are worth.

*(A few minutes each on a GPU: 20 epochs.)*

📖 Book: [§8.3.10 Training the U-Net](https://obooks.tech/imageprocessing/chapter8.html#sec-unet-training)

In [ ]:
torch.manual_seed(0)
unet_plain = SmallUNet().to(device)
print("Run 1 — U-Net, no augmentation")
history_plain = train(unet_plain, augment=False)

In [ ]:
torch.manual_seed(0)
unet_aug = SmallUNet().to(device)
print("Run 2 — U-Net, with augmentation")
history_aug = train(unet_aug, augment=True)

### 🧩 Puzzle 3 — Cut the skip connections

Without skips, the decoder only gets the upsampled maps.  How many input maps does `dec2` have now?
`dec1`?

<details><summary><b>💡 Hint</b> (click to open)</summary>

Puzzle 1 without the `+ 64` and `+ 32`.

</details>

In [ ]:
class NoSkipUNet(SmallUNet):
    def __init__(self):
        super().__init__()
        self.dec2 = ConvBlock(___, 64)
        self.dec1 = ConvBlock(___, 32)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b  = self.bottleneck(self.pool(s2))
        u2 = self.dec2(F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False))   # no torch.cat
        u1 = self.dec1(F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False))   # no torch.cat
        return self.out_conv(u1)

torch.manual_seed(0)
unet_noskip = NoSkipUNet().to(device)

# ── Check ──
assert unet_noskip(torch.zeros(1, 3, 256, 256, device=device)).shape == (1, 3, 256, 256), "Without skips, dec2 gets 128 maps and dec1 gets 64"
print("✅ Puzzle 3 solved!")

print("Run 3 — U-Net without skip connections, with augmentation")
history_noskip = train(unet_noskip, augment=True)

In [ ]:
plt.plot(range(1, 21), history_plain,  'o-', label='U-Net')
plt.plot(range(1, 21), history_aug,    'o-', label='U-Net + augmentation')
plt.plot(range(1, 21), history_noskip, 'o-', label='no skips + augmentation')
plt.axhline(0.812, color='gray', linestyle='--', label='Lab 4 (validation)')
plt.xlabel('epoch')
plt.ylabel('validation Dice')
plt.legend()
plt.show()

for name, h in [('U-Net', history_plain), ('U-Net + augmentation', history_aug), ('no skips + augmentation', history_noskip)]:
    print(f"{name:26s} best validation Dice {max(h):.4f} (epoch {int(np.argmax(h)) + 1})")

### 🤔 Think 2 — Reading the three curves

1. Which run reached the best validation Dice?  Did augmentation help, as the book predicts?
2. Look at the curves epoch by epoch.  Do they rise smoothly?  Why did we keep the *best* epoch rather
   than the last — and on which images did we judge "best"?
3. How much did the skip connections add?  Look at the masks below: what is different about the
   borders without them?
4. Compare with Lab 4 (0.812 on validation).  After nine labs, what has the U-Net bought us?

📖 Book: [§8.3.11 Why 0.77 — and How to Push It Higher](https://obooks.tech/imageprocessing/chapter8.html#why-0.77-and-how-to-push-it-higher)

In [ ]:
k = 0
i = val_idx[k]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(y[i], cmap='viridis', vmin=0, vmax=2)
axes[0].set_title(f'validation image {i}: truth')
for ax, (name, model) in zip(axes[1:], [('U-Net', unet_plain), ('+ augmentation', unet_aug), ('no skips', unet_noskip)]):
    pred = predict_masks(model, [i])[0]
    ax.imshow(pred, cmap='viridis', vmin=0, vmax=2)
    ax.set_title(f'{name}: Dice {image_dice(pred, y[i]):.3f}')
for ax in axes:
    ax.axis('off')
plt.show()

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. The Final Test — Once

We take the run with the best **validation** Dice to the test images, and save its weights so that
Lab 10 can use them.  (Lab 10 loads a copy trained with exactly this code, from the repository, so it
does not depend on this session.)

In [ ]:
runs = {'U-Net': (unet_plain, history_plain), 'U-Net + augmentation': (unet_aug, history_aug),
        'no skips + augmentation': (unet_noskip, history_noskip)}
best_name = max(runs, key=lambda name: max(runs[name][1]))
best_model = runs[best_name][0]
torch.save(best_model.state_dict(), 'unet_weights.pt')
print("Chosen on validation:", best_name, "— weights saved to unet_weights.pt")

test_preds = predict_masks(best_model, test_idx)
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])
nc_pred = np.array([nc_ratio(p) for p in test_preds])
dice  = mean_dice(test_preds, test_idx)
r2    = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
close = np.mean(np.abs(nc_pred - nc_true) < 0.05)

print()
print(f"{'':28s} test Dice   N/C R²   N/C within ±0.05")
print(f"{best_name:28s} {dice:.4f}    {r2:7.3f}   {close:.0%}")
print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 4 specks, fill              0.772     -0.465   35%")

plt.scatter(nc_true, nc_pred, alpha=0.7)
plt.plot([0, 2], [0, 2], 'r--')
plt.xlabel('true N/C ratio')
plt.ylabel('predicted N/C ratio')
plt.title('U-Net on the 40 test images (red line: perfect)')
plt.show()

### 🤔 Think 3 — Was it worth it?

1. How does the U-Net's test Dice compare with every earlier lab?  And its N/C R²?
2. The U-Net learns about 470,000 numbers from 120 images.  Name two things in this lab that
   protected it from simply memorising the training images.
3. Look at the scatter.  Are the N/C errors spread evenly, or are a few images badly off?  What would
   you look at next?

📖 Book: [§8.4.4 Evaluation and Failure Modes](https://obooks.tech/imageprocessing/chapter8.html#evaluation-and-failure-modes)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **A U-Net sees far and draws sharp.**  The encoder's downsampling widens the view to about 44 × 44
  pixels; the skip connections bring back the fine detail for exact borders.
- **The loss can target what you report.**  Soft Dice loss scores whole-region overlap; cross-entropy
  scores pixels.  Using both is common.
- **Augmentation makes more data out of the same images** — as long as image and mask get exactly the
  same transformation.
- **Watch validation during training** and keep the best epoch.  Training loss alone always looks good.
- **Ablations still apply to deep learning** — and can surprise you.  Removing the skip connections
  cost almost nothing on these blocky, hand-drawn labels.
- **Dice is not the N/C ratio.**  The best test Dice so far came with the worst N/C R².

---

### Group discussion 👥

Split the work, then share your numbers (GPU recommended):

| Person | Change, then re-run Run 2 |
|---|---|
| A | `epochs=40` |
| B | Augmentation without rotations (flips only) |
| C | `lr=1e-3` instead of `3e-4` |

1. Did training longer keep helping, or did validation level off?
2. Which part of the augmentation mattered more, flips or turns?
3. With the time you have, what single change would you try next to push the U-Net further?